# Supplementary Material C — Monte Carlo analysis

**Paper:** *Durability, not logistics, determines the climate benefit of jeans repair*  
**Journal:** *Sustainable Production and Consumption*  
**Authors:** [Author list]  
**DOI:** [Add DOI]  

---

This notebook reproduces all figures and statistical analyses reported in the paper.
Run all cells sequentially. Figures are saved to `figures/` and tables to `results/`.

**Prerequisites:** `pip install -r requirements.txt`  
**Data:** Place the Excel workbook (Supplementary Material B) in `data/`


In [ ]:
# Environment check
import sys
print(f'Python {sys.version}')
for pkg in ['pandas', 'numpy', 'scipy', 'matplotlib', 'seaborn', 'openpyxl']:
    mod = __import__(pkg)
    print(f'  {pkg}: {mod.__version__}')


In [ ]:
import os
os.makedirs('figures', exist_ok=True)
os.makedirs('figures/driver_plots', exist_ok=True)
os.makedirs('results', exist_ok=True)
print('Output directories ready.')


## 0. Setup — install / import libraries, define workbook path

In [ ]:
import pandas as pd
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import matplotlib.patches as mpatches
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

print('Libraries loaded OK')

In [ ]:
# ── Data path ──────────────────────────────────────────────────────
# The workbook (Supplementary Material B) should be in the data/ folder.
# If you placed it elsewhere, update this path.
WORKBOOK_PATH = 'data/Climate_calculations_MC_runs_frozen_260227.xlsx'

# Forest chart — Emissions per pair (lifecycle) vs per repair event (transport legs)

Compares the magnitude of lifecycle emission components (per pair of jeans)
against transport emission components (per repair event), all in kg CO₂e.

Data source : sheet MC_runs from the frozen Monte Carlo workbook.

## 0. Load data

In [ ]:
SHEET_NAME    = 'MC_runs'
HEADER_ROW    = 9

df_raw = pd.read_excel(
    WORKBOOK_PATH,
    sheet_name=SHEET_NAME,
    header=HEADER_ROW - 1,
    skiprows=[UNITS_ROW := 9],       # units row (0-indexed)
    engine='openpyxl',
)
df = df_raw[pd.to_numeric(df_raw['Iter'], errors='coerce').notna()].copy()
df = df.apply(pd.to_numeric, errors='coerce').dropna(subset=['Iter']).reset_index(drop=True)
df = df.drop(columns=[c for c in df.columns if 'Unnamed' in str(c)], errors='ignore')

print(f"Loaded {len(df)} iterations.")
# ── Derived columns (used by driver-explanation figures) ──────────
df['E_fixed'] = df['E_prod'] + df['E_collection'] + df['E_sorting'] + df['E_eol']
df['L_total'] = df['L_before_first_repair'] + df['N_repairs'] * df['ΔL_per_repair']


## 1. Compute emission components

In [ ]:
# --- 1a. Lifecycle emissions per pair ────────────────────────────────────
df['E_use_per_pair'] = (
    df['E_use'] * df['L_before_first_repair'] / df['Wears_per_wash']
)

lifecycle = {
    'E_prod':          'Production ($E_{prod}$)',
    'E_use_per_pair':  'Use phase ($E_{use}$ per pair)',
    'E_eol':           'End of life ($E_{eol}$)',
    'E_collection':    'Collection ($E_{coll}$)',
    'E_sorting':       'Sorting ($E_{sort}$)',
}

# --- 1b. Transport emissions per repair event ───────────────────────────
#     Customer legs: d × EF × AF
#     Van legs:      d × EF_van × mass

transport_legs = {}

# ── Tailor ───────────────────────────────────────────────────────────────
transport_legs['Tailor – walk/bike'] = (
    df['d_home_tailor_home'] * df['EF_walkORbike'] * df['AF_tailor_walkORbike']
)
transport_legs['Tailor – ICEV'] = (
    df['d_home_tailor_home'] * df['EF_car_ICEV'] * df['AF_tailor_car']
)
transport_legs['Tailor – EV'] = (
    df['d_home_tailor_home'] * df['EF_car_EL'] * df['AF_tailor_car']
)

# ── CWa door-to-door (van only) ─────────────────────────────────────────
transport_legs['CWa – van ICEV'] = (
    df['d_van_door_to_door'] * df['EF_van_ICEV'] * df['mass_pair_jeans']
)
transport_legs['CWa – van EV'] = (
    df['d_van_door_to_door'] * df['EF_van_EL'] * df['mass_pair_jeans']
)

# ── CWb box — customer legs ─────────────────────────────────────────────
transport_legs['CWb cust. – walk/bike'] = (
    df['d_cust_home_box_home'] * df['EF_walkORbike'] * df['AF_box_walkORbike']
)
transport_legs['CWb cust. – ICEV'] = (
    df['d_cust_home_box_home'] * df['EF_car_ICEV'] * df['AF_box_car']
)
transport_legs['CWb cust. – EV'] = (
    df['d_cust_home_box_home'] * df['EF_car_EL'] * df['AF_box_car']
)

# ── CWb box — van leg ───────────────────────────────────────────────────
transport_legs['CWb van – ICEV'] = (
    df['d_van_box_repairer'] * df['EF_van_ICEV'] * df['mass_pair_jeans']
)
transport_legs['CWb van – EV'] = (
    df['d_van_box_repairer'] * df['EF_van_EL'] * df['mass_pair_jeans']
)

# ── CWc postal — customer legs ──────────────────────────────────────────
transport_legs['CWc cust. – walk/bike'] = (
    df['d_cust_home_postal_home'] * df['EF_walkORbike'] * df['AF_postal_walkORbike']
)
transport_legs['CWc cust. – ICEV'] = (
    df['d_cust_home_postal_home'] * df['EF_car_ICEV'] * df['AF_postal_ICEV']
)
transport_legs['CWc cust. – EV'] = (
    df['d_cust_home_postal_home'] * df['EF_car_EL'] * df['AF_postal_EL']
)

# ── CWc postal — van leg ────────────────────────────────────────────────
transport_legs['CWc van – ICEV'] = (
    df['d_van_postal_repairer'] * df['EF_van_ICEV'] * df['mass_pair_jeans']
)
transport_legs['CWc van – EV'] = (
    df['d_van_postal_repairer'] * df['EF_van_EL'] * df['mass_pair_jeans']
)


## 2. Build summary table (Median, P5, P95)

In [ ]:
def summarise(series, label, group, colour):
    return {
        'label':  label,
        'group':  group,
        'colour': colour,
        'Median': series.median(),
        'P5':     series.quantile(0.05),
        'P95':    series.quantile(0.95),
    }

# ── Colour palette (harmonised with MC_analysis.ipynb) ──────────────────
C_LIFECYCLE  = '#555555'   # neutral grey for lifecycle
C_WALK       = '#2ca02c'   # green   — walk / bike
C_EV         = '#4575b4'   # blue    — EV
C_ICEV       = '#d73027'   # red     — ICEV
C_VAN_ICEV   = '#e08214'   # orange  — van ICEV
C_VAN_EV     = '#8073ac'   # purple  — van EV

rows = []

# Lifecycle
for col, label in lifecycle.items():
    rows.append(summarise(df[col], label, 'Lifecycle (per pair)', C_LIFECYCLE))

# Transport — assign colour by mode
mode_colour = {
    'walk/bike': C_WALK, 'ICEV': C_ICEV, 'EV': C_EV,
    'van ICEV':  C_VAN_ICEV, 'van EV': C_VAN_EV,
}

def _leg_colour(leg_name):
    low = leg_name.lower()
    if 'van' in low and 'ev' in low and 'icev' not in low:
        return C_VAN_EV
    if 'van' in low and 'icev' in low:
        return C_VAN_ICEV
    if 'walk' in low or 'bike' in low:
        return C_WALK
    if 'ev' in low and 'icev' not in low:
        return C_EV
    if 'icev' in low:
        return C_ICEV
    return '#888888'


# Group labels for transport sections
TRANSPORT_GROUPS = {
    'Tailor':       'Tailor – customer (per repair)',
    'CWa':          'CWa door-to-door – van (per repair)',
    'CWb cust.':    'CWb box – customer leg (per repair)',
    'CWb van':      'CWb box – van leg (per repair)',
    'CWc cust.':    'CWc postal – customer leg (per repair)',
    'CWc van':      'CWc postal – van leg (per repair)',
}

for leg_name, series in transport_legs.items():
    prefix = leg_name.split(' –')[0]
    group  = TRANSPORT_GROUPS.get(prefix, 'Transport (per repair)')
    rows.append(summarise(series, leg_name, group, _leg_colour(leg_name)))

summary = pd.DataFrame(rows)
print(summary[['label', 'group', 'P5', 'Median', 'P95']].to_string(index=False))


## Forest chart

In [ ]:
sns.set_theme(style='whitegrid')

n = len(summary)
y = np.arange(n)[::-1]          # top-to-bottom plotting

fig, ax = plt.subplots(figsize=(10, 0.48 * n + 2.0))

# ── Group separators ────────────────────────────────────────────────────
prev_group = summary['group'].iloc[0]
for idx in range(1, n):
    if summary['group'].iloc[idx] != prev_group:
        mid = (y[idx] + y[idx - 1]) / 2
        ax.axhline(mid, color='#aaaaaa', lw=0.6, zorder=1)
        prev_group = summary['group'].iloc[idx]

# ── Whiskers + dots ─────────────────────────────────────────────────────
for i in range(n):
    med = summary['Median'].iloc[i]
    lo  = summary['P5'].iloc[i]
    hi  = summary['P95'].iloc[i]
    col = summary['colour'].iloc[i]

    # Whisker (P5–P95)
    ax.plot([lo, hi], [y[i], y[i]], color=col, lw=1.8,
            solid_capstyle='round', zorder=3)
    cap = 0.18
    ax.plot([lo, lo], [y[i] - cap, y[i] + cap], color=col, lw=1.3, zorder=3)
    ax.plot([hi, hi], [y[i] - cap, y[i] + cap], color=col, lw=1.3, zorder=3)

    # Median dot
    ax.plot(med, y[i], 'o', color=col, ms=6.5,
            mec='white', mew=0.6, zorder=4)

    # Median label
    ax.text(med, y[i] + 0.32, f'{med:.3f}' if med < 1 else f'{med:.2f}',
            va='bottom', ha='center', fontsize=7,
            color='#333333', fontweight='bold')

# ── Group annotations (right margin) ───────────────────────────────────
prev_group = None
group_y = []
for idx in range(n):
    grp = summary['group'].iloc[idx]
    if grp != prev_group:
        if prev_group is not None and group_y:
            mid_y = np.mean(group_y)
            ax.text(1.02, mid_y, prev_group, transform=ax.get_yaxis_transform(),
                    va='center', ha='left', fontsize=7.5, color='#666666',
                    fontstyle='italic')
        group_y = [y[idx]]
        prev_group = grp
    else:
        group_y.append(y[idx])
# last group
if group_y:
    mid_y = np.mean(group_y)
    ax.text(1.02, mid_y, prev_group, transform=ax.get_yaxis_transform(),
            va='center', ha='left', fontsize=7.5, color='#666666',
            fontstyle='italic')

# ── Formatting ──────────────────────────────────────────────────────────
ax.set_yticks(y)
ax.set_yticklabels(summary['label'].values, fontsize=8.5)
ax.set_xscale('log')
ax.xaxis.set_major_formatter(
    ticker.FuncFormatter(lambda x, _: f'{x:g}'))
ax.set_xlabel('Emissions  (log$_{10}$ kg CO$_2$e)', fontsize=10)
ax.set_title(
    'Lifecycle emissions per pair  vs  transport emissions per repair event',
    fontsize=11, fontweight='bold', loc='left', pad=22,
)
ax.text(0.0, 1.015,
        'Median (dot) with P5–P95 interval across 1 000 MC draws',
        transform=ax.transAxes, fontsize=8, color='#666666',
        va='bottom', fontstyle='italic', ha='left')

ax.set_ylim(-0.8, y[0] + 1.8)
ax.grid(axis='x', alpha=0.25, which='major')
ax.grid(axis='y', visible=False)
ax.set_axisbelow(True)

# ── Legend ───────────────────────────────────────────────────────────────
legend_items = [
    mpatches.Patch(color=C_LIFECYCLE, label='Lifecycle'),
    mpatches.Patch(color=C_WALK,      label='Walk / bike'),
    mpatches.Patch(color=C_ICEV,      label='Car ICEV'),
    mpatches.Patch(color=C_EV,        label='Car EV'),
    mpatches.Patch(color=C_VAN_ICEV,  label='Van ICEV'),
    mpatches.Patch(color=C_VAN_EV,    label='Van EV'),
]
ax.legend(handles=legend_items, title='Emission source / mode',
          fontsize=7.5, title_fontsize=8,
          loc='lower right', framealpha=0.85)

plt.tight_layout()
plt.subplots_adjust(right=0.72)    # room for group annotations

# ── Save ────────────────────────────────────────────────────────────────
for ext in ('png', 'pdf'):
    fig.savefig(f'forest_emissions_per_pair.{ext}',
                dpi=200, bbox_inches='tight')
    print(f'Saved: forest_emissions_per_pair.{ext}')

plt.show()
plt.close(fig)

## Scenario comparison

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# Scenario comparison
# ═══════════════════════════════════════════════════════════════════════════
#
# Produces: Figure 2 — horizontal dot-and-whisker chart comparing
#           functional-unit emissions (kg CO₂e / 600 wears) across
#           5 repair pathway families + 1 linear baseline.
#
# Each pathway family aggregates its transport sub-scenarios:
#   - Thin whisker  = envelope P5–P95 (min P5 to max P95 across subs)
#   - Thick bar     = range of sub-scenario medians
#   - Central dot   = midpoint of the median range

sns.set_theme(style='whitegrid')

from matplotlib.lines import Line2D
from matplotlib.patches import Rectangle, ConnectionPatch
import matplotlib.gridspec as gridspec


FU_WEARS = 600

SCENARIOS = {
    'E_FU_reuse_no_transport':  'Self-repair',
    'E_FU_tailor_walk':         'Tailor',
    'E_FU_tailor_EL':           'Tailor',
    'E_FU_tailor_ICEV':         'Tailor',
    'E_FU_CWa_van_EL':         'CWa (door-to-door)',
    'E_FU_CWa_van_ICEV':       'CWa (door-to-door)',
    'E_FU_CWc_walk_van_EL':    'CWc (postal station)',
    'E_FU_CWc_walk_van_ICEV':  'CWc (postal station)',
    'E_FU_CWc_EL_van_EL':      'CWc (postal station)',
    'E_FU_CWc_EL_van_ICEV':    'CWc (postal station)',
    'E_FU_CWc_ICEV_van_EL':    'CWc (postal station)',
    'E_FU_CWc_ICEV_van_ICEV':  'CWc (postal station)',
    'E_FU_CWb_walk_van_EL':    'CWb (box station)',
    'E_FU_CWb_walk_van_ICEV':  'CWb (box station)',
    'E_FU_CWb_EL_van_EL':      'CWb (box station)',
    'E_FU_CWb_EL_van_ICEV':    'CWb (box station)',
    'E_FU_CWb_ICEV_van_EL':    'CWb (box station)',
    'E_FU_CWb_ICEV_van_ICEV':  'CWb (box station)',
}

FAMILY_ORDER = ['Self-repair', 'Tailor', 'CWa (door-to-door)',
                'CWc (postal station)', 'CWb (box station)']

# ── Statistics ─────────────────────────────────────────────────────────────
lin = df['E_FU_linear']
scen_rows = []
for col, fam in SCENARIOS.items():
    s = df[col]
    delta = s - lin
    scen_rows.append({
        'family': fam, 'mean': s.mean(), 'median': s.median(),
        'P5': s.quantile(0.05), 'P95': s.quantile(0.95),
        'p_repair_lt': (delta < 0).mean(), 'mean_delta': delta.mean(),
    })
scen_df = pd.DataFrame(scen_rows)

# ── Aggregate by pathway family ────────────────────────────────────────────
fam_rows = []
for fam in FAMILY_ORDER:
    sub = scen_df[scen_df['family'] == fam]
    fam_rows.append({
        'family': fam,
        'n_sub': len(sub),
        'envelope_P5':  sub['P5'].min(),
        'envelope_P95': sub['P95'].max(),
        'median_lo':    sub['median'].min(),
        'median_hi':    sub['median'].max(),
        'median_mid':   (sub['median'].min() + sub['median'].max()) / 2,
        'mean_delta_lo': sub['mean_delta'].min(),
        'mean_delta_hi': sub['mean_delta'].max(),
        'worst_p':       sub['p_repair_lt'].min(),
        'mean_lo':       sub['mean'].min(),
        'mean_hi':       sub['mean'].max(),
    })
fam_df = pd.DataFrame(fam_rows)

# ── Colours per pathway family ─────────────────────────────────────────────
FAMILY_COLOURS = {
    'Self-repair':          '#2ca02c',
    'Tailor':               '#1f77b4',
    'CWa (door-to-door)':   '#ff7f0e',
    'CWc (postal station)': '#9467bd',
    'CWb (box station)':    '#d62728',
}
C_LINEAR = '#888888'

# ── Helper ─────────────────────────────────────────────────────────────────
def draw_families(ax, fam_df, y_pos, thick_lw=6, dot_ms=8, cap=0.20,
                  thin_alpha=0.55, thick_alpha=0.85):
    for i, (_, r) in enumerate(fam_df.iterrows()):
        yi = y_pos[i]
        col = FAMILY_COLOURS[r['family']]
        # Thin envelope whisker
        ax.plot([r['envelope_P5'], r['envelope_P95']], [yi, yi],
                color=col, lw=1.2, solid_capstyle='round', zorder=3, alpha=thin_alpha)
        ax.plot([r['envelope_P5']]*2, [yi-cap, yi+cap],
                color=col, lw=1.0, zorder=3, alpha=thin_alpha)
        ax.plot([r['envelope_P95']]*2, [yi-cap, yi+cap],
                color=col, lw=1.0, zorder=3, alpha=thin_alpha)
        # Thick median-range bar
        if r['n_sub'] > 1:
            ax.plot([r['median_lo'], r['median_hi']], [yi, yi],
                    color=col, lw=thick_lw, solid_capstyle='round',
                    zorder=4, alpha=thick_alpha)
        # Central dot
        ax.plot(r['median_mid'], yi, 'o', color=col, ms=dot_ms,
                mec='white', mew=0.8, zorder=5)

# ══════════════════════════════════════════════════════════════════════════
# Stacked two-panel figure: (a) on top, (b) below
# ══════════════════════════════════════════════════════════════════════════
sns.set_theme(style='whitegrid')

n_fam = len(fam_df)
# Panel (a): 5 families + 1 linear = 6 rows
# Panel (b): 5 families only
y_a = np.arange(n_fam + 1)[::-1]
y_a_fam = y_a[:n_fam]
yi_lin = y_a[-1]
y_b = np.arange(n_fam)[::-1]

# Height ratios: panel (a) gets slightly more space because it has 6 rows
fig, (ax_a, ax_b) = plt.subplots(
    2, 1, figsize=(10, 8.0),
    gridspec_kw={'height_ratios': [1.15, 1], 'hspace': 0.32})

# ── Panel (a): Full-scale comparison ───────────────────────────────────────
lin_med = lin.median()
lin_p5  = lin.quantile(0.05)
lin_p95 = lin.quantile(0.95)

ax_a.axvspan(lin_p5, lin_p95, color='#cccccc', alpha=0.25, zorder=1)
ax_a.axvline(lin_med, color='#999999', ls='--', lw=1.2, zorder=2)

draw_families(ax_a, fam_df, y_a_fam)

# Annotations: sub-scenario count + median value for self-repair
for i, (_, r) in enumerate(fam_df.iterrows()):
    yi = y_a_fam[i]
    if r['n_sub'] == 1:
        ann = f"median {r['median_lo']:.1f}"
    else:
        ann = f"{r['n_sub']} sub-scenarios"
    ax_a.text(r['envelope_P95'] + 1.5, yi, ann,
              va='center', ha='left', fontsize=8,
              color='#555555', fontstyle='italic')

# Linear row
cap = 0.20
ax_a.plot([lin_p5, lin_p95], [yi_lin, yi_lin],
          color=C_LINEAR, lw=2.0, solid_capstyle='round', zorder=3)
ax_a.plot([lin_p5]*2, [yi_lin-cap, yi_lin+cap], color=C_LINEAR, lw=1.3, zorder=3)
ax_a.plot([lin_p95]*2, [yi_lin-cap, yi_lin+cap], color=C_LINEAR, lw=1.3, zorder=3)
ax_a.plot(lin_med, yi_lin, 'o', color=C_LINEAR, ms=8, mec='white', mew=0.8, zorder=5)
ax_a.text(lin_p95 + 1.5, yi_lin, f'{lin_med:.1f}',
          va='center', ha='left', fontsize=8.5, color='#333333', fontweight='bold')

# Separator
sep = (y_a_fam[-1] + yi_lin) / 2
ax_a.axhline(sep, color='#aaaaaa', lw=0.8, zorder=1)

# Panel (a) formatting
labels_a = list(fam_df['family']) + ['Linear (no repair)']
ax_a.set_yticks(y_a)
ax_a.set_yticklabels(labels_a, fontsize=9.5)
ax_a.set_xlabel('Functional-unit emissions (kg CO$_2$e / 600 wears)', fontsize=9.5)
ax_a.set_xlim(left=0, right=155)
ax_a.set_title('(a)  Full-scale comparison with linear baseline',
               fontsize=10.5, fontweight='bold', loc='left')

# Legend — now plenty of horizontal room to the right of the linear P95
legend_handles = [
    Line2D([0], [0], color='#999999', ls='--', lw=1.2, marker='None',
           label=f'Linear median ({lin_med:.1f})'),
    Line2D([0], [0], color='grey', lw=1.2, alpha=0.55,
           label='Envelope (P5\u2013P95 across sub-scenarios)'),
    Line2D([0], [0], color='grey', lw=6, alpha=0.6, solid_capstyle='round',
           label='Range of sub-scenario medians'),
    Line2D([0], [0], marker='o', color='grey', lw=0, ms=8, mec='white', mew=0.8,
           label='Midpoint of median range'),
]
ax_a.legend(handles=legend_handles, loc='right', fontsize=8,
            framealpha=0.92, edgecolor='#cccccc',
            bbox_to_anchor=(0.99, 0.5))

# ── Panel (b): Zoomed median detail ────────────────────────────────────────
draw_families(ax_b, fam_df, y_b, thick_lw=9, dot_ms=10)

# Median value labels
for i, (_, r) in enumerate(fam_df.iterrows()):
    yi = y_b[i]
    col = FAMILY_COLOURS[r['family']]
    if r['n_sub'] == 1:
        ax_b.text(r['median_mid'], yi + 0.28, f"{r['median_lo']:.1f}",
                  fontsize=9.5, color=col, fontweight='bold', ha='center', va='bottom')
    else:
        ax_b.text(r['median_lo'] - 0.15, yi, f"{r['median_lo']:.1f}",
                  fontsize=9.5, color=col, fontweight='bold', ha='right', va='center')
        ax_b.text(r['median_hi'] + 0.15, yi, f"{r['median_hi']:.1f}",
                  fontsize=9.5, color=col, fontweight='bold', ha='left', va='center')

# Panel (b) formatting
ax_b.set_xlim(28, 37.5)
ax_b.set_ylim(y_b[-1] - 0.6, y_b[0] + 0.6)
ax_b.set_yticks(y_b)
ax_b.set_yticklabels(list(fam_df['family']), fontsize=9.5)
ax_b.set_xlabel('Functional-unit emissions (kg CO$_2$e / 600 wears)', fontsize=9.5)
ax_b.set_title('(b)  Median detail across repair pathway families',
               fontsize=10.5, fontweight='bold', loc='left')
ax_b.grid(True, alpha=0.3)

# ── Zoom-source rectangle on panel (a) + connector lines to panel (b) ─────
# The rectangle highlights the x-range that panel (b) zooms into,
# covering only the 5 repair-family rows (not the linear row).
zoom_x = (28, 37.5)                          # must match ax_b.set_xlim
rect_bottom = y_a_fam[-1] - 0.5              # just below CWb
rect_top    = y_a_fam[0]  + 0.5              # just above Self-repair
rect = Rectangle(
    (zoom_x[0], rect_bottom),
    zoom_x[1] - zoom_x[0],                   # width
    rect_top - rect_bottom,                   # height
    linewidth=0.8, edgecolor='#999999', facecolor='#fafafa',
    alpha=0.35, zorder=1, linestyle='--')
ax_a.add_patch(rect)

# Connector lines from the bottom-left and bottom-right corners of the
# rectangle in panel (a) down to the top-left and top-right corners of
# panel (b). Using figure-level ConnectionPatch so the lines span axes.
for x_data, x_axes_b in [(zoom_x[0], 0), (zoom_x[1], 1)]:
    con = ConnectionPatch(
        # Start: bottom edge of rectangle in panel (a) data coords
        xyA=(x_data, rect_bottom), coordsA=ax_a.transData,
        # End: top edge of panel (b) in axes fraction coords
        xyB=(x_axes_b, 1), coordsB=ax_b.transAxes,
        color='#999999', lw=0.7, ls='--', alpha=0.45)
    fig.add_artist(con)

# ── Suptitle ───────────────────────────────────────────────────────────────
fig.suptitle(
    'Scenario comparison \u2014 functional-unit emissions by pathway family',
    fontsize=12, fontweight='bold', y=0.96)

plt.tight_layout(rect=[0, 0, 1, 0.96])  # leave room for suptitle


plt.tight_layout()
fig.savefig('figures/scenario_comparison.png', dpi=200, bbox_inches='tight')
fig.savefig('figures/scenario_comparison.pdf', bbox_inches='tight')
plt.show()

# ── Print condensed pathway table for in-text reference ────────────────────
print("\n" + "=" * 90)
print("CONDENSED PATHWAY TABLE")
print("=" * 90)
print(f"{'Pathway':<25} {'n':>3} {'Mean range':>14} {'Median range':>15} "
      f"{'Mean \u0394 range':>16} {'Reduction %':>13} {'Worst P':>9}")
print("-" * 90)
print(f"{'Linear (no repair)':<25} {'\u2013':>3} {lin.mean():>14.1f} {lin.median():>15.1f} "
      f"{'\u2013':>16} {'\u2013':>13} {'\u2013':>9}")
for _, r in fam_df.iterrows():
    n = r['n_sub']
    if n == 1:
        mean_str = f"{r['mean_lo']:.1f}"
        med_str  = f"{r['median_lo']:.1f}"
        delta_str = f"{r['mean_delta_lo']:.1f}"
        pct_str = f"{-r['mean_delta_lo']/lin.mean()*100:.0f}"
    else:
        mean_str = f"{r['mean_lo']:.1f}\u2013{r['mean_hi']:.1f}"
        med_str  = f"{r['median_lo']:.1f}\u2013{r['median_hi']:.1f}"
        delta_str = f"{r['mean_delta_lo']:.1f} to {r['mean_delta_hi']:.1f}"
        pct_lo = -r['mean_delta_hi'] / lin.mean() * 100
        pct_hi = -r['mean_delta_lo'] / lin.mean() * 100
        pct_str = f"{pct_lo:.0f}\u2013{pct_hi:.0f}"
    print(f"{r['family']:<25} {n:>3} {mean_str:>14} {med_str:>15} "
          f"{delta_str:>16} {pct_str:>13} {r['worst_p']:>9.1%}")
print("=" * 90)


# Break-even distance analysis

This section answers a critical practical question: 
**How far can each transport leg travel before the climate benefit of garment repair-and-reuse disappears?**

The break-even distance is the maximum one-way distance a transport leg can cover before the repair scenario emits more CO₂e than simply replacing the garment. It is derived by dividing the per-repair "transport budget" (emission savings from avoided production, minus non-transport repair costs) by the emission factor per km.

Because both terms are stochastic, the 1 000 Monte Carlo draws produce a full distribution of break-even distances. We report medians with 90 % credible intervals (P5–P95).

## 1. Load data from Excel

In [ ]:
# ─────────────────────────────────────────────────────────────────────────
SHEET_BE      = "MC_break_even"
BE_HEADER_ROW = 6        # row in Excel with column names
BE_FIRST_DATA = 7
BE_LAST_DATA  = 34
BE_COLS       = "A:F"    # Scenario … P95

CUST_ROWS = slice(0, 16)  # rows 7–22:  customer / single-leg distances
VAN_ROWS  = slice(16, 28) # rows 23–34: van-route distances
# ─────────────────────────────────────────────────────────────────────────

df_be = pd.read_excel(
    WORKBOOK_PATH,
    sheet_name=SHEET_BE,
    header=BE_HEADER_ROW - 1,
    usecols=BE_COLS,
    nrows=BE_LAST_DATA - BE_FIRST_DATA + 1,
    engine="openpyxl",
)
df_be = df_be.dropna(subset=["Scenario"]).reset_index(drop=True)
df_be[["Feasible share", "Median", "P5", "P95"]] = (
    df_be[["Feasible share", "Median", "P5", "P95"]]
    .apply(pd.to_numeric, errors="coerce")
)

print(f"Loaded {len(df_be)} break-even scenarios from '{SHEET_BE}'.")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# Nomenclature mapping
# ═══════════════════════════════════════════════════════════════════════
#
# Convention (consistent with the Spearman driver analysis):
#   Service model  :  Tailor / CWa / CWb / CWc
#   Customer mode  :  walk/bike / ICEV / EV           (not "EL")
#   Logistics mode :  van ICEV / van EV               (not "van EL")
#   Separator      :  en-dash " – "

BE_NAME_MAP = {
    "Tailor (customer ICEV)":                  "Tailor – customer ICEV",
    "Tailor (customer EL)":                    "Tailor – customer EV",
    "CWa door-to-door (van ICEV)":             "CWa door-to-door – van ICEV",
    "CWa door-to-door (van EL)":               "CWa door-to-door – van EV",
    "CWb box (cust walk/bike, van ICEV)":      "CWb box – walk/bike + van ICEV",
    "CWb box (cust walk/bike, van EL)":        "CWb box – walk/bike + van EV",
    "CWb box (cust ICEV, van ICEV)":           "CWb box – ICEV + van ICEV",
    "CWb box (cust ICEV, van EL)":             "CWb box – ICEV + van EV",
    "CWb box (cust EL, van ICEV)":             "CWb box – EV + van ICEV",
    "CWb box (cust EL, van EL)":               "CWb box – EV + van EV",
    "CWc postal (cust walk/bike, van ICEV)":   "CWc postal – walk/bike + van ICEV",
    "CWc postal (cust walk/bike, van EL)":     "CWc postal – walk/bike + van EV",
    "CWc postal (cust ICEV, van ICEV)":        "CWc postal – ICEV + van ICEV",
    "CWc postal (cust ICEV, van EL)":          "CWc postal – ICEV + van EV",
    "CWc postal (cust EL, van ICEV)":          "CWc postal – EV + van ICEV",
    "CWc postal (cust EL, van EL)":            "CWc postal – EV + van EV",
}

df_be["Scenario_h"] = (
    df_be["Scenario"]
    .map(BE_NAME_MAP)
    .fillna(df_be["Scenario"])
)


def _customer_mode(name):
    """Classify the customer's transport mode from the scenario name."""
    low = name.lower()
    if "walk/bike" in low:
        return "Walk / bike"
    elif "– customer ev" in low or "– ev +" in low:
        return "EV"
    elif "– customer icev" in low or "– icev +" in low:
        return "ICEV"
    elif "door-to-door" in low:
        return "Van only"
    return "Other"


def _service_model(name):
    """Extract the service-model prefix (Tailor / CWa / CWb / CWc)."""
    for prefix in ("Tailor", "CWa", "CWb", "CWc"):
        if name.startswith(prefix):
            return prefix
    return "Other"


df_be["cust_mode"] = df_be["Scenario_h"].apply(_customer_mode)
df_be["service"]   = df_be["Scenario_h"].apply(_service_model)

# Split into customer-leg and van-leg sub-frames
df_be_cust = df_be.iloc[CUST_ROWS].copy().reset_index(drop=True)
df_be_van  = df_be.iloc[VAN_ROWS].copy().reset_index(drop=True)

print(f"  Customer-leg scenarios : {len(df_be_cust)}")
print(f"  Van-route scenarios   : {len(df_be_van)}")

## 2.  Plot — break-even forest chart

In [ ]:
# ── Visual style (shared with Spearman driver section) ─────────────────
sns.set_theme(style="whitegrid")

# Colour palette — encodes the customer's transport mode
MODE_COLORS = {
    "Walk / bike": "#2ca02c",    # green  (zero-emission customer leg)
    "EV":          "#4575b4",    # blue   (low-emission)
    "ICEV":        "#d73027",    # red    (high-emission)
    "Van only":    "#ff7f0e",    # orange (CWa — no separate customer leg)
}

# Reference distances for real-world context
REF_DISTANCES = {
    5:  "Typical\nwalk / bike",
    20: "Urban\ncar trip",
    80: "Suburban\ncar trip",
}


def plot_be_panel(ax, df, title, subtitle=None, show_refs=True,
                  show_legend=True):
    """Forest-plot panel: median dot + 90 % CI whiskers,
    coloured by customer transport mode, grouped by service model."""
    
    n = len(df)
    y = np.arange(n)[::-1]

    # ── Service-model separators ────────────────────────────────
    for idx in range(1, n):
        if df["service"].iloc[idx] != df["service"].iloc[idx - 1]:
            ax.axhline((y[idx] + y[idx - 1]) / 2,
                       color="#aaaaaa", lw=0.6, zorder=1)

    # ── Plot each scenario ──────────────────────────────────────
    for i in range(n):
        med = df["Median"].iloc[i]
        lo  = df["P5"].iloc[i]
        hi  = df["P95"].iloc[i]
        col = MODE_COLORS.get(df["cust_mode"].iloc[i], "#888888")

        if not np.isfinite(med):
            continue

        # Whisker (P5–P95)
        ax.plot([lo, hi], [y[i], y[i]], color=col, lw=1.8,
                solid_capstyle="round", zorder=3)
        cap = 0.18
        ax.plot([lo, lo], [y[i] - cap, y[i] + cap], color=col,
                lw=1.3, zorder=3)
        ax.plot([hi, hi], [y[i] - cap, y[i] + cap], color=col,
                lw=1.3, zorder=3)
        # Median dot
        ax.plot(med, y[i], "o", color=col, ms=6.5,
                mec="white", mew=0.6, zorder=4)
        # Median label
        ax.text(med, y[i] + 0.32, f"{med:,.0f}",
                va="bottom", ha="center", fontsize=7,
                color="#333333", fontweight="bold")

    # ── Reference distance guides ───────────────────────────────
    if show_refs:
        for km, label in REF_DISTANCES.items():
            ax.axvline(km, color="#cccccc", lw=0.8, ls=":", zorder=1)
            ax.text(km, y[0] + 0.6, label, ha="center", va="bottom",
                    fontsize=6, color="#999999", fontstyle="italic")

    # ── Formatting ──────────────────────────────────────────────
    ax.set_yticks(y)
    ax.set_yticklabels(df["Scenario_h"].values, fontsize=8.5)
    ax.set_xscale("log")
    ax.xaxis.set_major_formatter(
        ticker.FuncFormatter(lambda x, _: f"{x:,.0f}"))
    ax.set_xlabel("Break-even distance (log$_{10}$ km)", fontsize=10)
    ax.set_title(title, fontsize=11, fontweight="bold", loc="left", pad=18)
    if subtitle:
        ax.text(0.0, 1.015, subtitle, transform=ax.transAxes,
                fontsize=8, color="#666666", va="bottom",
                fontstyle="italic", ha="left")
    ax.set_xlim(left=3)
    ax.set_ylim(-0.8, y[0] + 2.0)
    ax.grid(axis="x", alpha=0.25, which="major")
    ax.grid(axis="y", visible=False)
    ax.set_axisbelow(True)

    # ── Legend ───────────────────────────────────────────────────
    if show_legend:
        present = df["cust_mode"].unique()
        handles = [mpatches.Patch(color=MODE_COLORS[m], label=m)
                   for m in ("Walk / bike", "EV", "ICEV", "Van only")
                   if m in present]
        ax.legend(handles=handles, title="Customer mode",
                  fontsize=7.5, title_fontsize=8,
                  loc="lower right", framealpha=0.85)

In [ ]:
# ── Create the figure with two panels and plot the break-even results ───────
# The top panel shows the customer-leg break-even distances, and the bottom panel shows the van-route break-even distances. 
# The height of each panel is proportional to the number of scenarios it contains, to make efficient use of space while keeping 
# the scenario labels legible. Each scenario is plotted as a horizontal line (whisker) representing the 90 % confidence interval 
# (P5 to P95) of the break-even distance, with a dot at the median. 

fig, (ax1, ax2) = plt.subplots(
    2, 1, figsize=(10, 13),
    gridspec_kw={"height_ratios": [len(df_be_cust), len(df_be_van)]},
)

plot_be_panel(
    ax1, df_be_cust,
    title="(a)  Customer-leg break-even distance",
    subtitle=("Maximum one-way customer distance before repair "
              "loses its climate advantage"),
)

plot_be_panel(
    ax2, df_be_van,
    title="(b)  Van-route break-even distance",
    subtitle=("Maximum one-way van distance before repair "
              "loses its climate advantage"),
    show_legend=False,
)

plt.tight_layout(h_pad=3.5)

for ext in ("png", "pdf"):
    fig.savefig(f"break_even_forest.{ext}",
                dpi=300, bbox_inches="tight", facecolor="white")

print("Saved: break_even_forest.png  and  break_even_forest.pdf")
plt.show()

# Spearman driver analysis — Jeans repair Monte Carlo

This notebook reads the Monte Carlo simulation results from the Excel workbook,
computes Spearman rank correlations between key input parameters and the
**Δ = (repair scenario − linear)** outcome for each repair scenario,
and produces a ranked driver table and heatmap.

**Workbook sheet used:** `MC_runs` (rows 9–10 = headers/units, row 11 onwards = simulation iterations)

## 1. Load data from Excel

In [ ]:
# Data already loaded in section 1 above.
# Verify that df is available:
print(f'Using {len(df)} simulation iterations from shared data load.')
print(f'Columns available: {len(df.columns)}')


In [ ]:
# Drop any unnamed columns (artefacts from Excel import)
df = df[[c for c in df.columns if not c.startswith('Unnamed')]]


## Visualise imported data

In [ ]:

sns.set_theme(style="whitegrid")

# numeric columns except Iter
cols = [c for c in df.select_dtypes(include="number").columns if c != "Iter"]

# make pages so each subplot is large enough
plots_per_page = 16
ncols = 4
nrows = 4

for start in range(0, len(cols), plots_per_page):
    subcols = cols[start:start + plots_per_page]

    fig, axes = plt.subplots(nrows, ncols, figsize=(16, 14))
    axes = axes.flatten()

    for i, col in enumerate(subcols):
        ax = axes[i]
        s = df[col].dropna()

        # draw box first (clear edge, light fill)
        sns.boxplot(
            y=s,
            ax=ax,
            width=0.35,
            fliersize=0,
            color="#9ecae1",
            boxprops=dict(alpha=0.6, edgecolor="#1f4e79", linewidth=1.5),
            whiskerprops=dict(color="#1f4e79", linewidth=1.2),
            capprops=dict(color="#1f4e79", linewidth=1.2),
            medianprops=dict(color="#b22222", linewidth=1.6),
            zorder=2
        )

        # overlay jittered points
        sns.stripplot(
            y=s, 
            ax=ax,
            color="black",
            jitter=0.15, 
            alpha=0.35,
            size=2,
            zorder=3 
        )

        ax.set_title(col, fontsize=9)
        ax.set_xlabel("")
        ax.set_ylabel("")

        # optional: if no variation, make it explicit
        if s.nunique() <= 1:
            ax.text(0.5, 0.95, "no spread", transform=ax.transAxes,
                    ha="center", va="top", fontsize=8, color="crimson")

    # remove empty axes in last page
    for j in range(len(subcols), len(axes)):
        fig.delaxes(axes[j])

    plt.tight_layout()
    plt.show()




## 2. Define inputs and scenarios

In [ ]:
# ── Input (driver) parameters ────────────────────────────────────────────────
# These are the stochastic inputs we correlate against Δ
INPUTS = {
    # Lifecycle emissions
    'E_prod':               'Production emissions (kg CO₂e/pair)',
    'E_use':                'Use-phase emissions per wash (kg CO₂e/wash)',
    'Wears_per_wash':       'Wears per wash',
    # Reuse module
    'L_before_first_repair':'Lifespan before first repair (wears)',
    'ΔL_per_repair':        'Extra wears per repair',
    'N_repairs':            'Number of repairs',
    # Transport — distances
    'd_home_tailor_home':   'Tailor: customer distance (km/repair)',
    'd_van_door_to_door':   'CWa: van distance (km/repair)',
    'd_cust_home_box_home': 'CWb: customer → box distance (km/repair)',
    'd_van_box_repairer':   'CWb: van distance (km/repair)',
    'd_cust_home_postal_home': 'CWc: customer → postal distance (km/repair)',
    'd_van_postal_repairer':'CWc: van distance (km/repair)',
    # Physical
    'mass_pair_jeans':      'Jeans mass (kg)',
}

# ── Repair scenarios: column name → readable label ───────────────────────────
SCENARIOS = {
    'E_FU_reuse_no_transport':    'Self-repair (no transport)',
    'E_FU_tailor_walk':           'Tailor – walk/bike',
    'E_FU_tailor_ICEV':           'Tailor – ICEV car',
    'E_FU_tailor_EL':             'Tailor – EV car',
    'E_FU_CWa_van_ICEV':          'CWa door-to-door – van ICEV',
    'E_FU_CWa_van_EL':            'CWa door-to-door – van EV',
    'E_FU_CWb_walk_van_ICEV':     'CWb box – walk + van ICEV',
    'E_FU_CWb_walk_van_EL':       'CWb box – walk + van EV',
    'E_FU_CWb_ICEV_van_ICEV':     'CWb box – ICEV + van ICEV',
    'E_FU_CWb_ICEV_van_EL':       'CWb box – ICEV + van EV',
    'E_FU_CWb_EL_van_ICEV':       'CWb box – EV + van ICEV',
    'E_FU_CWb_EL_van_EL':         'CWb box – EV + van EV',
    'E_FU_CWc_walk_van_ICEV':     'CWc postal – walk + van ICEV',
    'E_FU_CWc_walk_van_EL':       'CWc postal – walk + van EV',
    'E_FU_CWc_ICEV_van_ICEV':     'CWc postal – ICEV + van ICEV',
    'E_FU_CWc_ICEV_van_EL':       'CWc postal – ICEV + van EV',
    'E_FU_CWc_EL_van_ICEV':       'CWc postal – EV + van ICEV',
    'E_FU_CWc_EL_van_EL':         'CWc postal – EV + van EV',
}

LINEAR_COL = 'E_FU_linear'

print('Setup complete.')
print(f'Drivers to test:  {len(INPUTS)}')
print(f'Repair scenarios: {len(SCENARIOS)}')

## 3. Compute Δ = repair − linear for each scenario

In [ ]:
delta_cols = {}
for col, label in SCENARIOS.items():
    delta_col = f'delta_{col}'
    df[delta_col] = df[col] - df[LINEAR_COL]
    delta_cols[col] = delta_col

print('Δ columns created:')
for col, delta_col in delta_cols.items():
    print(f'  {delta_col}: mean Δ = {df[delta_col].mean():.2f} kg CO₂e/FU')

## 4. Spearman rank correlations

In [ ]:
results = []

for scen_col, scen_label in SCENARIOS.items():
    delta_col = delta_cols[scen_col]
    y = df[delta_col].dropna()

    for inp_col, inp_label in INPUTS.items():
        if inp_col not in df.columns:
            continue
        x = df[inp_col]
        # Align on common non-null index
        mask = x.notna() & y.notna()
        if mask.sum() < 10:
            continue
        rho, pval = stats.spearmanr(x[mask], y[mask])
        results.append({
            'Scenario':   scen_label,
            'ScenarioCol': scen_col,
            'Input':      inp_label,
            'InputCol':   inp_col,
            'rho':        rho,
            'p_value':    pval,
            'abs_rho':    abs(rho),
        })

rdf = pd.DataFrame(results)
print(f'Computed {len(rdf)} correlations.')
rdf.head(10)

## 5. Top-driver table per scenario

In [ ]:
# For each scenario, show inputs ranked by |ρ|
pd.set_option('display.max_colwidth', 50)
pd.set_option('display.float_format', '{:.3f}'.format)

top_drivers = (
    rdf.sort_values(['Scenario', 'abs_rho'], ascending=[True, False])
       .groupby('Scenario')
       .head(5)
       [['Scenario', 'Input', 'rho', 'p_value']]
       .reset_index(drop=True)
)

top_drivers

## 6. Pivot table — ρ matrix (inputs × scenarios)

In [ ]:
rho_pivot = rdf.pivot_table(
    index='Input',
    columns='Scenario',
    values='rho'
)

# Sort inputs by mean |ρ| across all scenarios (most influential first)
rho_pivot['_mean_abs'] = rho_pivot.abs().mean(axis=1)
rho_pivot = rho_pivot.sort_values('_mean_abs', ascending=False).drop(columns='_mean_abs')

print('ρ matrix (rows = inputs, columns = repair scenarios):')
rho_pivot.round(3)

## Relevance filtering for distance parameters

Distance parameters (e.g. `d_van_door_to_door`) are only mechanistically active
in the scenarios where they enter the emission formula with a non-zero emission
factor. Correlating them against unrelated scenarios yields noise with random
sign, which distorts the mean ρ and the heatmap.

So, we have to define a relevance mapping and filter before averaging (tornado chart)
or mask irrelevant cells (heatmap).

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# RELEVANCE MAPPING — which distance is mechanistically active where?
# ═══════════════════════════════════════════════════════════════════════
#
# A distance parameter is "active" in a scenario only if it enters the
# emission formula AND is multiplied by a non-zero emission factor.
#
# Customer distances (d_home_tailor_home, d_cust_home_box_home,
# d_cust_home_postal_home) are active only when the customer drives
# (ICEV or EV), not when they walk/bike (zero-emission factor).
#
# Van distances are active in all scenarios of their service model
# (the van always has an emission factor > 0).
#
# Non-distance inputs (E_prod, E_use, Wears_per_wash, L₀, ΔL, N,
# mass) are mechanistically active in ALL repair scenarios.
# ═══════════════════════════════════════════════════════════════════════

# Map: input column → set of scenario columns where it's active
ACTIVE_SCENARIOS = {
    # Tailor customer distance: active only when customer drives
    'd_home_tailor_home': {
        'E_FU_tailor_ICEV',
        'E_FU_tailor_EL',
    },
    # CWa van distance: active in both CWa scenarios
    'd_van_door_to_door': {
        'E_FU_CWa_van_ICEV',
        'E_FU_CWa_van_EL',
    },
    # CWb customer → box distance: active when customer drives
    'd_cust_home_box_home': {
        'E_FU_CWb_ICEV_van_ICEV',
        'E_FU_CWb_ICEV_van_EL',
        'E_FU_CWb_EL_van_ICEV',
        'E_FU_CWb_EL_van_EL',
    },
    # CWb van distance: active in ALL CWb scenarios
    'd_van_box_repairer': {
        'E_FU_CWb_walk_van_ICEV',
        'E_FU_CWb_walk_van_EL',
        'E_FU_CWb_ICEV_van_ICEV',
        'E_FU_CWb_ICEV_van_EL',
        'E_FU_CWb_EL_van_ICEV',
        'E_FU_CWb_EL_van_EL',
    },
    # CWc customer → postal distance: active when customer drives
    'd_cust_home_postal_home': {
        'E_FU_CWc_ICEV_van_ICEV',
        'E_FU_CWc_ICEV_van_EL',
        'E_FU_CWc_EL_van_ICEV',
        'E_FU_CWc_EL_van_EL',
    },
    # CWc van distance: active in ALL CWc scenarios
    'd_van_postal_repairer': {
        'E_FU_CWc_walk_van_ICEV',
        'E_FU_CWc_walk_van_EL',
        'E_FU_CWc_ICEV_van_ICEV',
        'E_FU_CWc_ICEV_van_EL',
        'E_FU_CWc_EL_van_ICEV',
        'E_FU_CWc_EL_van_EL',
    },
}

# Non-distance inputs → active everywhere (no filtering needed)
# E_prod, E_use, Wears_per_wash, L_before_first_repair,
# ΔL_per_repair, N_repairs, mass_pair_jeans


# ═══════════════════════════════════════════════════════════════════════
# Helper: tag each row in rdf as "relevant" or not
# ═══════════════════════════════════════════════════════════════════════

def is_relevant(row):
    """Return True if this (input, scenario) pair is mechanistically active."""
    inp_col = row['InputCol']
    scen_col = row['ScenarioCol']
    if inp_col in ACTIVE_SCENARIOS:
        return scen_col in ACTIVE_SCENARIOS[inp_col]
    return True   # non-distance inputs are always active

rdf['relevant'] = rdf.apply(is_relevant, axis=1)

n_total    = len(rdf)
n_relevant = rdf['relevant'].sum()
n_masked   = n_total - n_relevant
print(f"Total (input × scenario) pairs:  {n_total}")
print(f"Mechanistically relevant:        {n_relevant}")
print(f"Masked as noise:                 {n_masked}")


## 7. Heatmap — masked for mechanistic relevance

Supporting information figure. Grey-hatched cells indicate distance–scenario
pairs where the distance does not enter the emission formula.

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# 7. Heatmap — masked for mechanistic relevance
# ═══════════════════════════════════════════════════════════════════════
#
# Cells where a distance parameter is not mechanistically active in a
# scenario are set to NaN and rendered as grey (hatched), so the reader
# sees only causally meaningful correlations.
# ═══════════════════════════════════════════════════════════════════════

import numpy as np

# ── Build the masked pivot ──────────────────────────────────────────
rho_pivot_raw = rdf.pivot_table(index='Input', columns='Scenario', values='rho')

# Build a matching boolean mask from rdf
relevance_pivot = rdf.pivot_table(index='Input', columns='Scenario',
                                   values='relevant', aggfunc='first')

# Apply: set irrelevant cells to NaN
rho_pivot_masked = rho_pivot_raw.where(relevance_pivot)

# Sort inputs by mean |ρ| across RELEVANT scenarios only
rho_pivot_masked['_mean_abs'] = rho_pivot_masked.abs().mean(axis=1)
rho_pivot_masked = rho_pivot_masked.sort_values('_mean_abs', ascending=False).drop(columns='_mean_abs')

# LaTeX formatting for readability
rho_pivot_hm = rho_pivot_masked.rename(index={
    "Production emissions (kg CO₂e/pair)": r"Production emissions (kg CO$_2$e/pair)",
    'Use-phase emissions per wash (kg CO₂e/wash)': r'Use-phase emissions per wash (kg CO$_2$e/wash)',
})

# ── Plot ────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(18, 6))

# Plot the heatmap (NaN cells will be left blank by seaborn)
sns.heatmap(
    rho_pivot_hm,
    ax=ax,
    cmap='RdBu_r',
    center=0,
    vmin=-1, vmax=1,
    annot=True,
    fmt='.2f',
    annot_kws={'size': 8},
    linewidths=0.4,
    linecolor='#cccccc',
    cbar_kws={'label': 'Spearman ρ  (Δ = repair − linear)'}
)

# Hatch the masked (NaN) cells with light grey cross-hatching
for i in range(rho_pivot_hm.shape[0]):
    for j in range(rho_pivot_hm.shape[1]):
        if np.isnan(rho_pivot_hm.iloc[i, j]):
            ax.add_patch(plt.Rectangle(
                (j, i), 1, 1,
                fill=True, facecolor='#f0f0f0',
                edgecolor='#cccccc', linewidth=0.4,
                hatch='///', zorder=0
            ))

ax.set_title(
    'Spearman rank correlations: driver inputs vs Δ emissions (repair − linear)\n'
    '(grey hatched = distance not mechanistically active in scenario)',
    fontsize=12, pad=14
)
ax.set_xlabel('')
ax.set_ylabel('Input parameter', fontsize=10)
ax.tick_params(axis='x', rotation=45, labelsize=8)
ax.tick_params(axis='y', labelsize=9)

plt.setp(ax.get_xticklabels(), ha='right', rotation=45, rotation_mode='anchor')

plt.tight_layout()
plt.savefig('figures/spearman_heatmap_filtered.png', dpi=150, bbox_inches='tight')
plt.savefig('figures/spearman_heatmap_filtered.pdf', bbox_inches='tight')
plt.show()
print('Figure saved as spearman_heatmap_filtered.png / .pdf')


### Note on negative ρ for CWc: customer → postal distance

In the heatmap above, `CWc: customer → postal distance` shows small **negative**
ρ values (−0.02 to −0.00) in its four mechanistically active scenarios
(CWc ICEV/EV × van ICEV/EV). This sign is counterintuitive — mechanistically,
a longer customer trip should *add* transport emissions to the repair pathway,
making Δ less negative (positive ρ). The cause is a **confounding artefact from
finite Monte Carlo sampling**, not a model error.

#### Mechanism

In the frozen 1,000-draw sample, `d_cust_home_postal_home` and
`L_before_first_repair` happen to carry a small spurious correlation
(Spearman ρ ≈ −0.04). Because the parameters are drawn independently, this is
expected sampling noise — with *n* = 1,000 the standard error of ρ under the
null is approximately 1/√(*n* − 1) ≈ 0.032.

However, L₀ is the **dominant** driver of Δ (ρ ≈ +0.81). The spurious link
creates an indirect confounding pathway:

```
d_postal ──(−0.04)──▸ L₀ ──(+0.81)──▸ Δ
         spurious         true causal
```

The indirect effect is approximately (−0.04) × (+0.81) ≈ **−0.033**, which
overwhelms the tiny true direct effect of d_postal on Δ (≈ +0.02 to +0.05).

#### Confirmation via partial correlation

Computing partial Spearman ρ (controlling for L₀) recovers the correct positive
sign in all four active scenarios:

| Scenario | Raw ρ | Partial ρ (controlling for L₀) |
|----------|------:|------:|
| CWc – ICEV + van ICEV | −0.003 | **+0.051** |
| CWc – ICEV + van EV   | −0.003 | **+0.050** |
| CWc – EV + van ICEV   | −0.020 | **+0.022** |
| CWc – EV + van EV     | −0.020 | **+0.021** |

#### Why only this distance?

The customer → postal distance has the **smallest range** of all distance
parameters (a few kilometres to the nearest post office), producing the weakest
direct effect on Δ. This makes it uniquely vulnerable to being overwhelmed by
the confounding pathway through L₀. Other distances (Tailor customer, CWb
customer → box) have wider ranges and/or higher emission factors, giving them
enough direct signal to maintain the correct positive sign despite similar
sampling noise.

#### Implication

At |ρ| ≈ 0.01, this parameter is well below the negligible-influence threshold
(|ρ| < 0.04) and is excluded from the tornado chart in the paper. The sign
anomaly has no bearing on the study's conclusions.

## 8. Tornado chart — driver ranking (paper figure)

Only drivers with mean |ρ| > 0.04 across relevant scenarios are shown.
Below this threshold, correlations are indistinguishable from Monte Carlo
sampling noise (≈ 1/√*n* ≈ 0.032 for *n* = 1,000) and their sign may
reflect spurious inter-parameter correlations rather than causal structure
(see the CWc postal distance note above).

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# 8. Tornado chart — driver ranking by mean Spearman ρ
# ═══════════════════════════════════════════════════════════════════════
#
# Only drivers with mean |ρ| > 0.04 (across relevant scenarios) are
# shown.  Below this threshold, correlations are indistinguishable
# from Monte Carlo sampling noise (see explanation cell below the
# heatmap for details).
#
# For distance parameters, the average is taken only over scenarios
# where the distance is mechanistically active (relevance filter).
# For non-distance parameters, all 18 repair scenarios contribute.
# ═══════════════════════════════════════════════════════════════════════

RHO_THRESHOLD = 0.04   # minimum mean |ρ| to display

# LaTeX formatting (idempotent if already applied)
rdf["Input"] = rdf["Input"].str.replace("CO₂e", r"CO$_2$e", regex=False)

# ── Filter to relevant pairs only ───────────────────────────────────
rdf_rel = rdf[rdf['relevant']].copy()

# Mean ρ (signed) and mean |ρ| across relevant scenarios
mean_rho = rdf_rel.groupby('Input')['rho'].mean()
mean_abs = rdf_rel.groupby('Input')['abs_rho'].mean()

# ── Apply threshold ─────────────────────────────────────────────────
keep = mean_abs[mean_abs > RHO_THRESHOLD].index
dropped = mean_abs[mean_abs <= RHO_THRESHOLD].index

mean_rho = mean_rho.reindex(keep)
mean_abs = mean_abs.reindex(keep)

# Sort by |ρ|: smallest at bottom (barh plots bottom-up)
sort_order = mean_abs.sort_values(ascending=True).index
mean_rho = mean_rho.reindex(sort_order)

# Colour: red for positive ρ (less benefit), blue for negative ρ (more benefit)
colors = ['#d73027' if v > 0 else '#4575b4' for v in mean_rho.values]

# ── Plot ────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 5))

bars = ax.barh(
    y=range(len(mean_rho)),
    width=mean_rho.values,
    color=colors,
    edgecolor='white',
    linewidth=0.5,
    height=0.7,
)

# Y-axis: driver labels
ax.set_yticks(range(len(mean_rho)))
ax.set_yticklabels(mean_rho.index, fontsize=9)

# X-axis
ax.set_xlabel('Mean Spearman ρ across relevant repair scenarios', fontsize=10)

# Symmetric x-limits for visual balance
x_max = max(abs(mean_rho.min()), abs(mean_rho.max())) * 1.15
ax.set_xlim(-x_max, x_max)

# Centre line
ax.axvline(0, color='black', lw=0.8)

# Reference threshold lines (mirrored)
for thresh, ls in [(0.1, '--'), (0.3, ':')]:
    ax.axvline( thresh, color='grey', lw=0.7, ls=ls, alpha=0.6)
    ax.axvline(-thresh, color='grey', lw=0.7, ls=ls, alpha=0.6)

# ── Directional annotations ────────────────────────────────────────
y_top = len(mean_rho) - 0.2
ax.text(-x_max * 0.55, y_top + 0.5,
        '← higher value = more benefit from repair',
        ha='center', va='bottom', fontsize=10, fontstyle='italic',
        color='#4575b4')
ax.text( x_max * 0.55, y_top + 0.5,
        'higher value = less benefit from repair →',
        ha='center', va='bottom', fontsize=10, fontstyle='italic',
        color='#d73027')

ax.set_title(
    'Driver ranking — influence on Δ emissions (repair − linear)',
    fontsize=11, pad=40, fontweight="bold"
)

# ── Colour legend ───────────────────────────────────────────────────
from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor='#4575b4', label='Negative ρ  (repair more beneficial)'),
    Patch(facecolor='#d73027', label='Positive ρ  (repair less beneficial)'),
]
ax.legend(handles=legend_elements, fontsize=8, loc='lower right')

# ── Value labels on bars ────────────────────────────────────────────
for i, (val, bar) in enumerate(zip(mean_rho.values, bars)):
    offset = x_max * 0.02
    ha = 'left' if val >= 0 else 'right'
    x_pos = val + offset if val >= 0 else val - offset
    ax.text(x_pos, i, f'{val:+.2f}', va='center', ha=ha,
            fontsize=7.5, color='#333333')

sns.despine(left=True)
ax.tick_params(axis='y', length=0)

plt.tight_layout()
plt.savefig('figures/spearman_tornado.png', dpi=150, bbox_inches='tight')
plt.savefig('figures/spearman_tornado.pdf', bbox_inches='tight')
plt.show()
print('Figure saved as spearman_tornado.png / .pdf')

# ── Summary ─────────────────────────────────────────────────────────
print(f"\nDrivers shown (mean |ρ| > {RHO_THRESHOLD}):")
for inp in mean_abs.sort_values(ascending=False).index:
    sub = rdf_rel[rdf_rel['Input'] == inp]
    print(f"  {inp:<50} ρ = {mean_rho[inp]:>+.3f}   |ρ| = {mean_abs[inp]:.3f}   (n = {len(sub)})")

print(f"\nDrivers omitted (mean |ρ| ≤ {RHO_THRESHOLD}):")
mean_abs_all = rdf_rel.groupby('Input')['abs_rho'].mean()
mean_rho_all = rdf_rel.groupby('Input')['rho'].mean()
for inp in dropped:
    sub = rdf_rel[rdf_rel['Input'] == inp]
    print(f"  {inp:<50} ρ = {mean_rho_all[inp]:>+.3f}   |ρ| = {mean_abs_all[inp]:.3f}   (n = {len(sub)})")


## 9. Scatter plots — top 4 drivers for one focal scenario

In [ ]:
# Choose a focal scenario to illustrate
FOCAL_SCENARIO_COL = 'E_FU_CWb_ICEV_van_ICEV'   # change if desired
FOCAL_LABEL = SCENARIOS[FOCAL_SCENARIO_COL]

focal = rdf[rdf['ScenarioCol'] == FOCAL_SCENARIO_COL].sort_values('abs_rho', ascending=False)
top4 = focal.head(4)

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
axes = axes.flatten()

delta_col = delta_cols[FOCAL_SCENARIO_COL]
y = df[delta_col]

for ax, (_, row) in zip(axes, top4.iterrows()):
    x = df[row['InputCol']]
    mask = x.notna() & y.notna()
    ax.scatter(x[mask], y[mask], alpha=0.3, s=10, color='steelblue')
    ax.set_xlabel(row['Input'], fontsize=9)
    ax.set_ylabel('Δ (kg CO$_2$e/FU)', fontsize=9)
    ax.set_title(f"ρ = {row['rho']:.3f}", fontsize=10)
    ax.axhline(0, color='red', lw=0.8, ls='--')
    ax.tick_params(labelsize=8)

fig.suptitle(f'Top 4 drivers — {FOCAL_LABEL}\nΔ = repair emissions − linear', fontsize=11)
plt.tight_layout()
plt.savefig('figures/spearman_scatter.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure saved as spearman_scatter.png')

## 10. Export results to Excel

In [ ]:
OUTPUT_FILE = 'results/spearman_results.xlsx'

with pd.ExcelWriter(OUTPUT_FILE, engine='openpyxl') as writer:
    # Full correlation table
    rdf[['Scenario','Input','rho','p_value']].round(4).to_excel(
        writer, sheet_name='All_correlations', index=False
    )
    # Pivot matrix
    rho_pivot.round(3).to_excel(writer, sheet_name='Rho_matrix')
    # Top 5 per scenario
    top_drivers.round(3).to_excel(writer, sheet_name='Top5_per_scenario', index=False)

print(f'Results written to {OUTPUT_FILE}')

---
## Interpretation guide

| ρ value | Interpretation |
|---------|----------------|
| ρ < 0   | Higher input → Δ more negative → **repair more beneficial** |
| ρ > 0   | Higher input → Δ less negative → **repair less beneficial** |
| \|ρ\| > 0.5 | Strong driver |
| 0.2 < \|ρ\| < 0.5 | Moderate driver |
| \|ρ\| < 0.2 | Weak / negligible influence |

**Expected pattern for this model:**
- `N_repairs` and `ΔL_per_repair` should have strong **negative** ρ — more/longer repairs extend lifetime and avoid more production.
- `E_prod` should also have strong negative ρ — higher production emissions per pair make repair more valuable.
- Transport distances should have weak positive ρ for scenarios that involve long logistics legs.
- `E_use` and `Wears_per_wash` are expected to be weak because they affect both pathways similarly.

# Explaining the mechanisms behind Lifespan before first repair and Production emissions per pair 

In [ ]:
# ── Derived columns needed for the driver-explanation figure ────────────────
df['E_fixed'] = df['E_prod'] + df['E_collection'] + df['E_sorting'] + df['E_eol']
df['delta_reuse'] = df['E_FU_reuse_no_transport'] - df['E_FU_linear']
df['pw_fixed_lin'] = df['E_fixed'] / df['L_before_first_repair']
df['pw_fixed_rep'] = df['E_fixed'] / df['L_total']

sns.set_theme(style='white')

fig, axes = plt.subplots(2, 2, figsize=(12, 10))

ROLLING = 50
E_fixed_med = df['E_fixed'].median()

# ═══════════════════════════════════════════════════════════════════════════
# (a) L₀ mechanism: converging hyperbolas + scatter clouds
# ═══════════════════════════════════════════════════════════════════════════
ax = axes[0, 0]
L0_range = np.linspace(50, 300, 300)
N_typ, dL_typ = 3, 80
L_total_range = L0_range + N_typ * dL_typ

ax.scatter(df['L_before_first_repair'], df['pw_fixed_lin'],
           alpha=0.15, s=10, color='#d73027', edgecolors='none', zorder=1)
ax.scatter(df['L_before_first_repair'], df['pw_fixed_rep'],
           alpha=0.15, s=10, color='#4575b4', edgecolors='none', zorder=1)

lin_curve = E_fixed_med / L0_range
rep_curve = E_fixed_med / L_total_range
ax.plot(L0_range, lin_curve, color='#d73027', lw=2.5, zorder=3,
        label='Linear: $E_{fixed}$ / $L_0$')
ax.plot(L0_range, rep_curve, color='#4575b4', lw=2.5, zorder=3,
        label='Repair: $E_{fixed}$ / $L$')
ax.fill_between(L0_range, rep_curve, lin_curve,
                alpha=0.10, color='#4575b4', zorder=2)

ax.annotate('Large gap\n(big savings)', xy=(70, 0.09), fontsize=9,
            color='#4575b4', ha='center', fontstyle='italic')
ax.annotate('Small gap\n(small savings)', xy=(255, 0.028), fontsize=9,
            color='#4575b4', ha='center', fontstyle='italic')

ax.set_xlabel('$L_0$ (wears before first repair)', fontsize=10)
ax.set_ylabel('Per-wear fixed emissions\n(kg CO$_2$e / wear)', fontsize=10)
ax.set_title('(a) Mechanism: savings shrink as $L_0$ rises',
             fontsize=11, fontweight='bold')
ax.legend(fontsize=8, loc='upper right')
ax.set_xlim(50, 300)
ax.set_ylim(0, 0.25)

# ═══════════════════════════════════════════════════════════════════════════
# (b) Empirical Δ vs L₀
# ═══════════════════════════════════════════════════════════════════════════
ax = axes[0, 1]
ax.scatter(df['L_before_first_repair'], df['delta_reuse'],
           alpha=0.25, s=12, color='steelblue', edgecolors='none')
ax.axhline(0, color='red', lw=1, ls='--', alpha=0.7)

s1 = df.sort_values('L_before_first_repair')
roll1 = s1['delta_reuse'].rolling(ROLLING, center=True).median()
ax.plot(s1['L_before_first_repair'], roll1,
        color='#d73027', lw=2.5, label=f'Rolling median (n = {ROLLING})')

ax.set_xlabel('$L_0$ (wears before first repair)', fontsize=10)
ax.set_ylabel('$\\Delta$ = $E_{FU,repair}$ − $E_{FU,linear}$\n'
              '(kg CO$_2$e / 600 wears)', fontsize=10)
ax.set_title('(b) Empirical: $L_0$ vs $\\Delta$ (self-repair)',
             fontsize=11, fontweight='bold')
ax.legend(fontsize=8)
ax.set_xlim(50, 300)

# ═══════════════════════════════════════════════════════════════════════════
# (c) E_prod mechanism: diverging lines + scatter clouds
# ═══════════════════════════════════════════════════════════════════════════
ax = axes[1, 0]
E_other_med = (df['E_collection'] + df['E_sorting'] + df['E_eol']).median()
L0_med = df['L_before_first_repair'].median()
L_med  = df['L_total'].median()

Eprod_range = np.linspace(3, 21, 300)
E_fixed_range = Eprod_range + E_other_med

ax.scatter(df['E_prod'], df['pw_fixed_lin'],
           alpha=0.15, s=10, color='#d73027', edgecolors='none', zorder=1)
ax.scatter(df['E_prod'], df['pw_fixed_rep'],
           alpha=0.15, s=10, color='#4575b4', edgecolors='none', zorder=1)

lin_curve2 = E_fixed_range / L0_med
rep_curve2 = E_fixed_range / L_med
ax.plot(Eprod_range, lin_curve2, color='#d73027', lw=2.5, zorder=3,
        label=f'Linear: $E_{{fixed}}$ / median $L_0$')
ax.plot(Eprod_range, rep_curve2, color='#4575b4', lw=2.5, zorder=3,
        label=f'Repair: $E_{{fixed}}$ / median $L$')
ax.fill_between(Eprod_range, rep_curve2, lin_curve2,
                alpha=0.10, color='#4575b4', zorder=2)

ax.annotate('Small gap', xy=(5, 0.03), fontsize=9,
            color='#4575b4', ha='center', fontstyle='italic')
ax.annotate('Large gap', xy=(18, 0.10), fontsize=9,
            color='#4575b4', ha='center', fontstyle='italic')

ax.set_xlabel('$E_{prod}$ (kg CO$_2$e / pair)', fontsize=10)
ax.set_ylabel('Per-wear fixed emissions\n(kg CO$_2$e / wear)', fontsize=10)
ax.set_title('(c) Mechanism: savings grow as $E_{prod}$ rises',
             fontsize=11, fontweight='bold')
ax.legend(fontsize=8, loc='upper left')
ax.set_xlim(3, 21)
ax.set_ylim(0, 0.17)

# ═══════════════════════════════════════════════════════════════════════════
# (d) Empirical Δ vs E_prod
# ═══════════════════════════════════════════════════════════════════════════
ax = axes[1, 1]
ax.scatter(df['E_prod'], df['delta_reuse'],
           alpha=0.25, s=12, color='steelblue', edgecolors='none')
ax.axhline(0, color='red', lw=1, ls='--', alpha=0.7)

s2 = df.sort_values('E_prod')
roll2 = s2['delta_reuse'].rolling(ROLLING, center=True).median()
ax.plot(s2['E_prod'], roll2,
        color='#d73027', lw=2.5, label=f'Rolling median (n = {ROLLING})')

ax.set_xlabel('$E_{prod}$ (kg CO$_2$e / pair)', fontsize=10)
ax.set_ylabel('$\\Delta$ = $E_{FU,repair}$ − $E_{FU,linear}$\n'
              '(kg CO$_2$e / 600 wears)', fontsize=10)
ax.set_title('(d) Empirical: $E_{prod}$ vs $\\Delta$ (self-repair)',
             fontsize=11, fontweight='bold')
ax.legend(fontsize=8)

for row in axes:
    for a in row:
        a.grid(False)

plt.tight_layout()
plt.savefig('figures/driver_explanation_L0_Eprod.png', dpi=150, bbox_inches='tight')
plt.savefig('figures/driver_explanation_L0_Eprod.pdf', bbox_inches='tight')
plt.show()
print('Figure saved as driver_explanation_L0_Eprod.png / .pdf')


# For the graphical abstract

In [ ]:

# ── Colors ───────────────────────────────────────────────────────────────────
C_LINEAR = '#E83C63'
C_REPAIR = '#009BA5'
C_AXIS   = '#BFBFBF'

def _strip_axes(ax):
    """Remove all text/ticks/spines except the L-shaped left + bottom axes."""
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    for spine in ('left', 'bottom'):
        ax.spines[spine].set_color(C_AXIS)
        ax.spines[spine].set_linewidth(2.5)
    ax.set_xticks([])
    ax.set_yticks([])
    ax.tick_params(left=False, bottom=False)
    ax.set_xlabel('')
    ax.set_ylabel('')
    ax.set_title('')

def _save(fig, stem):
    fig.tight_layout()
    fig.savefig(f'figures//Graphical_abstract/{stem}.svg',
                format='svg', bbox_inches='tight', transparent=True)
    fig.savefig(f'figures//Graphical_abstract/{stem}.png',
                dpi=300, bbox_inches='tight', transparent=True)
    plt.show()
    plt.close(fig)
    print(f'Saved: {stem}.svg / .png')


# ── Shared medians ────────────────────────────────────────────────────────────
df['E_fixed']      = df['E_prod'] + df['E_collection'] + df['E_sorting'] + df['E_eol']
df['pw_fixed_lin'] = df['E_fixed'] / df['L_before_first_repair']
df['pw_fixed_rep'] = df['E_fixed'] / df['L_total']

E_fixed_med  = df['E_fixed'].median()
E_other_med  = (df['E_collection'] + df['E_sorting'] + df['E_eol']).median()
L0_med       = df['L_before_first_repair'].median()
L_med        = df['L_total'].median()
N_typ        = 3
dL_typ       = 80

# ═══════════════════════════════════════════════════════════════════════════════
# Panel (a)  –  L₀ mechanism   (converging hyperbolas)
# ═══════════════════════════════════════════════════════════════════════════════
fig, ax = plt.subplots(figsize=(4, 3.2))

L0_range       = np.linspace(50, 300, 300)
L_total_range  = L0_range + N_typ * dL_typ
lin_curve      = E_fixed_med / L0_range
rep_curve      = E_fixed_med / L_total_range

ax.scatter(df['L_before_first_repair'], df['pw_fixed_lin'],
           alpha=0.30, s=10, color=C_LINEAR, edgecolors='none', zorder=1)
ax.scatter(df['L_before_first_repair'], df['pw_fixed_rep'],
           alpha=0.30, s=10, color=C_REPAIR, edgecolors='none', zorder=1)
ax.plot(L0_range, lin_curve, color=C_LINEAR, lw=2.5, zorder=3)
ax.plot(L0_range, rep_curve, color=C_REPAIR, lw=2.5, zorder=3)
ax.fill_between(L0_range, rep_curve, lin_curve, alpha=0.12, color=C_REPAIR, zorder=2)

ax.set_xlim(50, 300)
ax.set_ylim(0, 0.25)
_strip_axes(ax)
plt.show()
#_save(fig, 'panel_a_L0')

# ═══════════════════════════════════════════════════════════════════════════════
# Panel (b)  –  E_prod mechanism   (diverging lines)
# ═══════════════════════════════════════════════════════════════════════════════
fig, ax = plt.subplots(figsize=(4, 3.2))

Eprod_range   = np.linspace(3, 21, 300)
E_fixed_range = Eprod_range + E_other_med
lin_curve2    = E_fixed_range / L0_med
rep_curve2    = E_fixed_range / L_med

ax.scatter(df['E_prod'], df['pw_fixed_lin'],
           alpha=0.30, s=10, color=C_LINEAR, edgecolors='none', zorder=1)
ax.scatter(df['E_prod'], df['pw_fixed_rep'],
           alpha=0.30, s=10, color=C_REPAIR, edgecolors='none', zorder=1)
ax.plot(Eprod_range, lin_curve2, color=C_LINEAR, lw=2.5, zorder=3)
ax.plot(Eprod_range, rep_curve2, color=C_REPAIR, lw=2.5, zorder=3)
ax.fill_between(Eprod_range, rep_curve2, lin_curve2, alpha=0.12, color=C_REPAIR, zorder=2)

ax.set_xlim(3, 21)
ax.set_ylim(0, 0.17)
_strip_axes(ax)
plt.show()
#_save(fig, 'panel_b_Eprod')

# ═══════════════════════════════════════════════════════════════════════════════
# Panel (c)  –  N mechanism
# Linear per-wear cost is FLAT in N (N doesn't affect L₀).
# Repair curve falls as N increases: more repairs → longer L_total.
# ═══════════════════════════════════════════════════════════════════════════════
fig, ax = plt.subplots(figsize=(4, 3.2))

N_range    = np.linspace(1, 8, 300)
lin_flatN  = np.full_like(N_range, E_fixed_med / L0_med)    # flat horizontal
rep_curveN = E_fixed_med / (L0_med + N_range * dL_typ)      # hyperbola

ax.scatter(df['N_repairs'], df['pw_fixed_lin'],
           alpha=0.30, s=10, color=C_LINEAR, edgecolors='none', zorder=1)
ax.scatter(df['N_repairs'], df['pw_fixed_rep'],
           alpha=0.30, s=10, color=C_REPAIR, edgecolors='none', zorder=1)
ax.plot(N_range, lin_flatN,   color=C_LINEAR, lw=2.5, zorder=3)
ax.plot(N_range, rep_curveN,  color=C_REPAIR, lw=2.5, zorder=3)
ax.fill_between(N_range, rep_curveN, lin_flatN, alpha=0.12, color=C_REPAIR, zorder=2)

ax.set_xlim(1, 8)
ax.set_ylim(0, None)
_strip_axes(ax)
plt.show()
#_save(fig, 'panel_c_N')

# ═══════════════════════════════════════════════════════════════════════════════
# Panel (d)  –  ΔL mechanism
# Same structure as N: linear is flat, repair falls as ΔL grows.
# ═══════════════════════════════════════════════════════════════════════════════
fig, ax = plt.subplots(figsize=(4, 3.2))

dL_range    = np.linspace(20, 200, 300)
lin_flatdL  = np.full_like(dL_range, E_fixed_med / L0_med)   # flat horizontal
rep_curvedL = E_fixed_med / (L0_med + N_typ * dL_range)      # hyperbola

ax.scatter(df['ΔL_per_repair'], df['pw_fixed_lin'],
           alpha=0.30, s=10, color=C_LINEAR, edgecolors='none', zorder=1)
ax.scatter(df['ΔL_per_repair'], df['pw_fixed_rep'],
           alpha=0.30, s=10, color=C_REPAIR, edgecolors='none', zorder=1)
ax.plot(dL_range, lin_flatdL,  color=C_LINEAR, lw=2.5, zorder=3)
ax.plot(dL_range, rep_curvedL, color=C_REPAIR, lw=2.5, zorder=3)
ax.fill_between(dL_range, rep_curvedL, lin_flatdL, alpha=0.12, color=C_REPAIR, zorder=2)

ax.set_xlim(20, 200)
ax.set_ylim(0, None)
_strip_axes(ax)
plt.show()
#_save(fig, 'panel_d_dL')

# Extra. Driver explanation plots — functional-unit (600 wears)
For every driver shown in the Spearman bar chart, this notebook
produces a three-panel figure:

 **(a)** Mechanistic panel — tailored to the driver category:

   - **Lifetime drivers** (L₀, N, ΔL): pairs of jeans consumed
     under linear vs repair, showing how the "pairs avoided" gap
     responds to the driver.
   - **Per-pair emission drivers** (E_prod, E_eol, etc.): per-pair
     emission cost × pairs consumed, showing how the "emission gap
     per pair saved" responds to the driver.
   - **Transport drivers** (distances, mass): cumulative transport
     emissions added per FU, showing the cost that erodes the
     repair benefit.
   - **Wash drivers** (E_use, Wears_per_wash): per-wear wash
     emissions, which affect both pathways equally and therefore
     cancel in Δ — panel (a) demonstrates this cancellation.

 **(b)** Empirical Δ = E_FU,repair − E_FU,linear scatter with
     rolling median and Spearman ρ annotation.

 **(c)** FU-level savings (= −Δ) scatter with rolling median.

Panel (a) now shows the actual mechanism at work for each driver category:
**Lifetime drivers (L₀, N_repairs, ΔL_per_repair)** show pairs of jeans consumed under each pathway, with the shaded gap representing pairs avoided. For L₀, you see the converging hyperbolas — the gap narrows because the linear pathway is already efficient at high L₀. For N_repairs and ΔL_per_repair, you see the repair curve dropping as more/longer repairs push L_total up, widening the gap and saving more pairs.

**Per-pair emission drivers (E_prod)** show FU-level fixed emissions (pairs × E_fixed) for each pathway. Both curves rise with E_prod, but the linear curve rises faster because it consumes more pairs — so the shaded gap widens. This is why higher production emissions make repair more beneficial (ρ = −0.47): when each pair is expensive in CO₂e terms, avoiding surplus pairs through repair saves more.

**Transport drivers (all distances plus jeans mass)** show the transport overhead that repair adds, computed as E_FU,scenario − E_FU,self-repair. This isolates the transport cost from the lifetime benefit, making it clear that transport is a tax on the repair pathway that grows with distance, eroding the savings. The weak ρ values (all < 0.1) are visually evident — the transport overhead is small relative to the scatter driven by other parameters.

**Wash drivers (E_use, Wears_per_wash)** show that per-FU wash emissions are identical in both pathways — a single overlapping cloud. This directly demonstrates why these parameters have near-zero ρ: they cancel in Δ because both pathways deliver the same 600 wears with the same washing frequency.


In [ ]:

sns.set_theme(style='whitegrid')
# Data already loaded in section 1. Derived columns added here.
# (E_fixed, pairs_linear, pairs_repair, pairs_saved are computed above.)

# %% Driver configuration
# ─────────────────────────────────────────────────────────────────────
# Each driver has:
#   col            : column name in MC_runs
#   label          : x-axis display label
#   scenario_col   : E_FU column for the repair pathway
#   scenario_label : human-readable scenario name
#   category       : one of "lifetime", "per_pair", "transport", "wash"
#                    — controls what panel (a) shows
# ─────────────────────────────────────────────────────────────────────

DRIVERS = [
    {
        "col": "L_before_first_repair",
        "label": "Lifespan before first repair (wears)",
        "scenario_col": "E_FU_reuse_no_transport",
        "scenario_label": "Self-repair (no transport)",
        "category": "lifetime",
    },
    {
        "col": "E_prod",
        "label": "Production emissions (kg CO$_2$e/pair)",
        "scenario_col": "E_FU_reuse_no_transport",
        "scenario_label": "Self-repair (no transport)",
        "category": "per_pair",
    },
    {
        "col": "N_repairs",
        "label": "Number of repairs",
        "scenario_col": "E_FU_reuse_no_transport",
        "scenario_label": "Self-repair (no transport)",
        "category": "lifetime",
    },
    {
        "col": "\u0394L_per_repair",
        "label": "Extra wears per repair",
        "scenario_col": "E_FU_reuse_no_transport",
        "scenario_label": "Self-repair (no transport)",
        "category": "lifetime",
    },
    {
        "col": "mass_pair_jeans",
        "label": "Jeans mass (kg)",
        "scenario_col": "E_FU_CWa_van_ICEV",
        "scenario_label": "CWa door-to-door \u2013 van ICEV",
        "category": "transport",
    },
    {
        "col": "d_home_tailor_home",
        "label": "Tailor: customer distance (km/repair)",
        "scenario_col": "E_FU_tailor_ICEV",
        "scenario_label": "Tailor \u2013 ICEV car",
        "category": "transport",
    },
    {
        "col": "d_cust_home_postal_home",
        "label": "CWc: customer \u2192 postal distance (km/repair)",
        "scenario_col": "E_FU_CWc_walk_van_ICEV",
        "scenario_label": "CWc postal \u2013 walk + van ICEV",
        "category": "transport",
    },
    {
        "col": "d_van_box_repairer",
        "label": "CWb: van distance (km/repair)",
        "scenario_col": "E_FU_CWb_walk_van_ICEV",
        "scenario_label": "CWb box \u2013 walk + van ICEV",
        "category": "transport",
    },
    {
        "col": "d_cust_home_box_home",
        "label": "CWb: customer \u2192 box distance (km/repair)",
        "scenario_col": "E_FU_CWb_ICEV_van_ICEV",
        "scenario_label": "CWb box \u2013 ICEV + van ICEV",
        "category": "transport",
    },
    {
        "col": "d_van_door_to_door",
        "label": "CWa: van distance (km/repair)",
        "scenario_col": "E_FU_CWa_van_ICEV",
        "scenario_label": "CWa door-to-door \u2013 van ICEV",
        "category": "transport",
    },
    {
        "col": "E_use",
        "label": "Use-phase emissions per wash (kg CO$_2$e/wash)",
        "scenario_col": "E_FU_reuse_no_transport",
        "scenario_label": "Self-repair (no transport)",
        "category": "wash",
    },
    {
        "col": "Wears_per_wash",
        "label": "Wears per wash",
        "scenario_col": "E_FU_reuse_no_transport",
        "scenario_label": "Self-repair (no transport)",
        "category": "wash",
    },
    {
        "col": "d_van_postal_repairer",
        "label": "CWc: van distance (km/repair)",
        "scenario_col": "E_FU_CWc_walk_van_ICEV",
        "scenario_label": "CWc postal \u2013 walk + van ICEV",
        "category": "transport",
    },
]

print(f"Configured {len(DRIVERS)} drivers.")

# %% Panel (a) rendering functions — one per category
# ─────────────────────────────────────────────────────────────────────

ROLLING_WINDOW = 50


def _panel_a_lifetime(ax, tmp, xlabel, scen_label):
    """Lifetime drivers (L₀, N, ΔL):
    Show pairs of jeans consumed under each pathway.
    The gap = pairs avoided = the physical source of savings."""

    # Pairs consumed per FU for each draw
    # (already in df as pairs_linear / pairs_repair, but we need
    #  them in the sorted tmp frame)
    pairs_lin = FU_WEARS / tmp["L_before_first_repair"]
    pairs_rep = FU_WEARS / tmp["L_total"]

    ax.scatter(tmp["x"], pairs_lin, alpha=0.15, s=10,
               color="#d73027", edgecolors="none", label="Linear: 600 / $L_0$")
    ax.scatter(tmp["x"], pairs_rep, alpha=0.15, s=10,
               color="#4575b4", edgecolors="none", label="Repair: 600 / $L$")

    # Rolling medians
    roll_lin = pairs_lin.rolling(ROLLING_WINDOW, center=True).median()
    roll_rep = pairs_rep.rolling(ROLLING_WINDOW, center=True).median()
    ax.plot(tmp["x"], roll_lin, color="#d73027", lw=2.5)
    ax.plot(tmp["x"], roll_rep, color="#4575b4", lw=2.5)

    # Shade the gap
    ax.fill_between(tmp["x"], roll_rep, roll_lin,
                    alpha=0.12, color="#4575b4", label="Pairs avoided")

    ax.set_ylabel("Pairs of jeans consumed\nper 600 wears", fontsize=10)
    ax.set_title("(a) Pairs consumed per FU", fontsize=11, fontweight="bold")
    ax.legend(fontsize=8, loc="best", markerscale=2)


def _panel_a_per_pair(ax, tmp, xlabel, scen_label):
    """Per-pair emission drivers (E_prod):
    Show FU-level fixed emissions = pairs × E_fixed for each pathway.
    The gap shows how higher E_fixed amplifies the benefit of having
    fewer pairs (repair) vs more pairs (linear)."""

    fu_fixed_lin = (FU_WEARS / tmp["L_before_first_repair"]) * tmp["E_fixed"]
    fu_fixed_rep = (FU_WEARS / tmp["L_total"]) * tmp["E_fixed"]

    ax.scatter(tmp["x"], fu_fixed_lin, alpha=0.15, s=10,
               color="#d73027", edgecolors="none",
               label="Linear: pairs $\\times$ $E_{fixed}$")
    ax.scatter(tmp["x"], fu_fixed_rep, alpha=0.15, s=10,
               color="#4575b4", edgecolors="none",
               label="Repair: pairs $\\times$ $E_{fixed}$")

    roll_lin = fu_fixed_lin.rolling(ROLLING_WINDOW, center=True).median()
    roll_rep = fu_fixed_rep.rolling(ROLLING_WINDOW, center=True).median()
    ax.plot(tmp["x"], roll_lin, color="#d73027", lw=2.5)
    ax.plot(tmp["x"], roll_rep, color="#4575b4", lw=2.5)

    ax.fill_between(tmp["x"], roll_rep, roll_lin,
                    alpha=0.12, color="#4575b4", label="Emission gap (= savings)")

    ax.set_ylabel("FU fixed emissions\n(kg CO$_2$e / 600 wears)", fontsize=10)
    ax.set_title("(a) Fixed emissions per FU: gap widens\n"
                 "with per-pair cost",
                 fontsize=11, fontweight="bold")
    ax.legend(fontsize=8, loc="best", markerscale=2)


def _panel_a_transport(ax, tmp, xlabel, scen_label):
    """Transport drivers (distances, mass):
    Show the transport-emission overhead added to the repair pathway
    per FU.  The linear pathway has zero transport, so all transport
    emissions directly erode the repair benefit."""

    # Transport overhead = E_FU,repair_scenario − E_FU,reuse_no_transport
    # This isolates the transport component from the lifetime benefit.
    transport_overhead = tmp["e_rep"] - tmp["e_rep_no_transport"]

    ax.scatter(tmp["x"], transport_overhead, alpha=0.2, s=10,
               color="#e66101", edgecolors="none")

    roll = transport_overhead.rolling(ROLLING_WINDOW, center=True).median()
    ax.plot(tmp["x"], roll, color="#d73027", lw=2.5,
            label=f"Rolling median (n = {ROLLING_WINDOW})")

    ax.set_ylabel("Transport overhead per FU\n"
                  "(kg CO$_2$e / 600 wears)", fontsize=10)
    ax.set_title("(a) Transport cost eroding\n"
                 "repair benefit",
                 fontsize=11, fontweight="bold")
    ax.legend(fontsize=8, loc="best")


def _panel_a_wash(ax, tmp, xlabel, scen_label):
    """Wash drivers (E_use, Wears_per_wash):
    Show per-wear wash emissions for both pathways.  Because e_wash
    is identical in linear and repair, the two clouds overlap
    perfectly — demonstrating why wash parameters cancel in Δ."""

    # Per-wear wash cost = E_use / Wears_per_wash (same for both pathways)
    wash_per_wear = tmp["E_use"] / tmp["Wears_per_wash"]

    # FU-level wash cost (identical for both pathways)
    wash_fu = FU_WEARS * wash_per_wear

    ax.scatter(tmp["x"], wash_fu, alpha=0.2, s=10,
               color="#7570b3", edgecolors="none",
               label="Both pathways (identical)")

    roll = wash_fu.rolling(ROLLING_WINDOW, center=True).median()
    ax.plot(tmp["x"], roll, color="#d73027", lw=2.5,
            label=f"Rolling median (n = {ROLLING_WINDOW})")

    ax.set_ylabel("FU wash emissions\n"
                  "(kg CO$_2$e / 600 wears)", fontsize=10)
    ax.set_title("(a) Wash cost: identical in both\n"
                 "pathways \u2192 cancels in \u0394",
                 fontsize=11, fontweight="bold")
    ax.legend(fontsize=8, loc="best")


# Map category names to rendering functions
PANEL_A_FUNCS = {
    "lifetime":  _panel_a_lifetime,
    "per_pair":  _panel_a_per_pair,
    "transport": _panel_a_transport,
    "wash":      _panel_a_wash,
}


# %% Main plotting function
# ─────────────────────────────────────────────────────────────────────

def plot_driver(driver_cfg, df, save_dir="."):
    """Generate a 3-panel FU-level explanation figure for one driver."""

    col            = driver_cfg["col"]
    xlabel         = driver_cfg["label"]
    scen_col       = driver_cfg["scenario_col"]
    scen_label     = driver_cfg["scenario_label"]
    category       = driver_cfg["category"]

    # ── Build working frame sorted by the driver ────────────────
    x     = df[col]
    e_lin = df["E_FU_linear"]
    e_rep = df[scen_col]
    delta = e_rep - e_lin
    savings = -delta

    # Spearman rho
    mask = x.notna() & delta.notna()
    rho, pval = stats.spearmanr(x[mask], delta[mask])

    # Sorted frame with all columns needed by any panel-a function
    tmp = pd.DataFrame({
        "x": x, "delta": delta, "savings": savings,
        "e_lin": e_lin, "e_rep": e_rep,
        # columns needed by specific panel-a functions:
        "L_before_first_repair": df["L_before_first_repair"],
        "L_total": df["L_total"],
        "E_fixed": df["E_fixed"],
        "E_use": df["E_use"],
        "Wears_per_wash": df["Wears_per_wash"],
        "e_rep_no_transport": df["E_FU_reuse_no_transport"],
    }).dropna(subset=["x", "delta"]).sort_values("x").reset_index(drop=True)

    roll_delta   = tmp["delta"].rolling(ROLLING_WINDOW, center=True).median()
    roll_savings = tmp["savings"].rolling(ROLLING_WINDOW, center=True).median()

    # ── Figure ──────────────────────────────────────────────────
    fig, axes = plt.subplots(1, 3, figsize=(17, 5.5))

    # ── Panel (a): category-specific mechanistic view ───────────
    panel_a_func = PANEL_A_FUNCS[category]
    panel_a_func(axes[0], tmp, xlabel, scen_label)
    axes[0].set_xlabel(xlabel, fontsize=10)

    # ── Panel (b): Δ scatter ────────────────────────────────────
    ax = axes[1]
    ax.scatter(tmp["x"], tmp["delta"], alpha=0.2, s=10,
               color="steelblue", edgecolors="none")
    ax.axhline(0, color="red", lw=1, ls="--", alpha=0.7)
    ax.plot(tmp["x"], roll_delta, color="#d73027", lw=2.5,
            label=f"Rolling median (n = {ROLLING_WINDOW})")
    ax.set_xlabel(xlabel, fontsize=10)
    ax.set_ylabel("$\\Delta$ = $E_{FU,repair}$ \u2212 $E_{FU,linear}$\n"
                  "(kg CO$_2$e / 600 wears)", fontsize=10)
    p_str = "p < 0.001" if pval < 0.001 else f"p = {pval:.3f}"
    ax.set_title(f"(b) $\\Delta$ vs driver   [\u03c1 = {rho:.3f}, {p_str}]",
                 fontsize=11, fontweight="bold")
    ax.legend(fontsize=8)

    # ── Panel (c): FU savings scatter ───────────────────────────
    ax = axes[2]
    ax.scatter(tmp["x"], tmp["savings"], alpha=0.2, s=10,
               color="#2ca02c", edgecolors="none")
    ax.axhline(0, color="red", lw=1, ls="--", alpha=0.7)
    ax.plot(tmp["x"], roll_savings, color="#d73027", lw=2.5,
            label=f"Rolling median (n = {ROLLING_WINDOW})")
    ax.set_xlabel(xlabel, fontsize=10)
    ax.set_ylabel("FU savings from repair\n"
                  "(kg CO$_2$e / 600 wears)", fontsize=10)
    ax.set_title("(c) FU savings vs driver", fontsize=11,
                 fontweight="bold")
    ax.legend(fontsize=8)

    # ── Suptitle and save ───────────────────────────────────────
    sign_text = ("higher \u2192 repair LESS beneficial" if rho > 0
                 else "higher \u2192 repair MORE beneficial")
    fig.suptitle(
        f"Driver: {xlabel}\n"
        f"Scenario: {scen_label}   |   "
        f"Spearman \u03c1 = {rho:+.3f}   |   "
        f"Sign: {sign_text}",
        fontsize=11, y=1.04
    )
    plt.tight_layout()

    safe_name = col.replace("\u0394", "delta")
    fname = f"{save_dir}/driver_{safe_name}_FU.png"
    fig.savefig(fname, dpi=150, bbox_inches="tight")
    plt.show()
    plt.close(fig)
    return fname, rho


# %% Generate all figures
# ─────────────────────────────────────────────────────────────────────

SAVE_DIR = "figures/driver_plots"
os.makedirs(SAVE_DIR, exist_ok=True)

print(f"\nGenerating {len(DRIVERS)} driver figures ...\n")
print(f"{'Driver':<50} {'Cat':<10} {'rho':>8}  File")
print("-" * 100)

for dcfg in DRIVERS:
    fname, rho = plot_driver(dcfg, df, save_dir=SAVE_DIR)
    print(f"{dcfg['label']:<50} {dcfg['category']:<10} {rho:>+8.3f}  {fname}")

print(f"\nAll figures saved to ./{SAVE_DIR}/")